# Projet Deep Learning for Computer Vision : Entraînement

**Auteurs :** Olivier BOROT, Marin NAGY  
**Date :** Février 2026

Ce notebook présente le pipeline complet d'entraînement pour la détection de bâtiments sur images satellitaires.

## Objectifs
1.  Préparer les données (chargement, split Train/Val/Test).
2.  Configurer le modèle YOLOv5 Nano (Transfer Learning).
3.  Entraîner le modèle sur le jeu d'entraînement.
4.  Valider les performances à chaque époque.
5.  Sauvegarder le meilleur modèle pour l'évaluation finale.

---

## 1. Importations et Configuration

In [47]:
import os
import glob
import json
import random
import yaml
import time
from pathlib import Path
from copy import deepcopy

import numpy as np

# PATCH: NumPy 2.0+ supprime np.trapz (utilisé par yolov5.utils.metrics)
if not hasattr(np, 'trapz'):
    np.trapz = np.trapezoid

import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm

import torch
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

# Import YOLOv5 modules
import yolov5
from yolov5.models.yolo import Model
from yolov5.utils.general import check_yaml, yaml_load, intersect_dicts, one_cycle, non_max_suppression
from yolov5.utils.loss import ComputeLoss
from yolov5.utils.metrics import box_iou, ap_per_class
from yolov5.utils.torch_utils import select_device, de_parallel

# Fixer la seed pour la reproductibilité
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

# Configuration
DATA_DIR = Path('../dataset')
IMG_SIZE = 640
BATCH_SIZE = 8
EPOCHS = 100   # 100 époques (~40 min) : la loss ne plateau pas à 50
DEVICE = select_device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")


YOLOv5  2026-3-2 Python-3.13.1 torch-2.10.0+cpu CPU



Using device: cpu


## 2. Chargement des Données

### 2.1. Chargement et Mélange des Images

In [48]:
# Lister toutes les images du dataset
images_dir = DATA_DIR / 'train' / 'images'
all_images = glob.glob(str(images_dir / '*.jpg')) + glob.glob(str(images_dir / '*.png'))

# Mélanger de façon reproductible (seed déjà fixée via set_seed(42))
random.shuffle(all_images)

total_images = len(all_images)
print(f"Nombre total d'images trouvées : {total_images}")
print(f"Exemples : {[os.path.basename(p) for p in all_images[:3]]}")

Nombre total d'images trouvées : 144
Exemples : ['image_142_png.rf.4260cd68ef718c1017d8e18069c389c5.jpg', 'image_8_png.rf.9f4d32fdc7f2b9b4b82dbe23f01851e2.jpg', 'image_53_png.rf.45e8de9518c63bd59a4ea758aa06a1e5.jpg']


### 2.2. Split du Dataset (80% Train / 10% Val / 10% Test)

Nous divisons manuellement les images en trois ensembles :
*   **Train (80%)** : Pour l'apprentissage des poids.
*   **Validation (10%)** : Pour le monitoring et le choix du meilleur modèle.
*   **Test (10%)** : Pour l'évaluation finale (jamais vu durant l'entraînement).

Cette séparation est faite de manière aléatoire (grâce au `shuffle` précédent) mais déterministe (grâce au `seed`).

In [49]:
class YoloDetectionDataset(Dataset):
    """
    Dataset personnalisé pour YOLOv5.
    Charge les images et les labels au format YOLO (class_id x y w h).
    Augmentation : flip horizontal + flip vertical (pertinent pour imagerie satellite).
    """
    def __init__(self, image_paths, label_dir, img_size=640, augment=False):
        self.image_paths = image_paths
        self.label_dir = label_dir
        self.img_size = img_size
        self.augment = augment

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label_filename = os.path.basename(img_path).replace('.jpg', '.txt').replace('.png', '.txt')
        label_path = os.path.join(self.label_dir, label_filename)

        try:
            img = Image.open(img_path).convert('RGB')
        except Exception as e:
            print(f"Erreur chargement image {img_path}: {e}")
            return torch.zeros((3, self.img_size, self.img_size)), torch.zeros((0, 6))

        img = img.resize((self.img_size, self.img_size))
        img_np = np.array(img)

        targets = np.zeros((0, 5))
        if os.path.exists(label_path):
            if os.path.getsize(label_path) > 0:
                l = np.loadtxt(label_path)
                if len(l.shape) == 1:
                    l = l.reshape(1, -1)
                targets = l.copy()

        # Augmentation géométrique (flip H + flip V, prob 0.5 chacun)
        # Pertinent pour l'imagerie satellite : pas d'orientation préférentielle
        if self.augment:
            if random.random() < 0.5:  # Flip horizontal
                img_np = np.fliplr(img_np).copy()
                if len(targets):
                    targets[:, 1] = 1.0 - targets[:, 1]
            if random.random() < 0.5:  # Flip vertical
                img_np = np.flipud(img_np).copy()
                if len(targets):
                    targets[:, 2] = 1.0 - targets[:, 2]

        img_tensor = torch.from_numpy(img_np).permute(2, 0, 1).float() / 255.0
        targets = torch.tensor(targets, dtype=torch.float32)

        return img_tensor, targets


def collate_fn(batch):
    """Gère un batch d'images et de targets de tailles variables."""
    imgs, targets = list(zip(*batch))
    new_targets = []
    for i, t in enumerate(targets):
        if t.numel() > 0:
            img_idx = torch.full((t.shape[0], 1), i)
            new_targets.append(torch.cat((img_idx, t), 1))

    if new_targets:
        targets = torch.cat(new_targets, 0)
    else:
        targets = torch.zeros((0, 6))

    return torch.stack(imgs), targets


In [50]:
# Split 80 / 10 / 10 (Train / Val / Test)
train_split = int(0.8 * total_images)
val_split = int(0.9 * total_images)

train_imgs = all_images[:train_split]
val_imgs = all_images[train_split:val_split]
test_imgs = all_images[val_split:]

In [51]:
# Création des DataLoaders
labels_dir = str(DATA_DIR / 'train/labels')

train_dataset = YoloDetectionDataset(train_imgs, labels_dir, img_size=IMG_SIZE, augment=True)
val_dataset   = YoloDetectionDataset(val_imgs,   labels_dir, img_size=IMG_SIZE, augment=False)

# WeightedRandomSampler : sur-échantillonnage des images contenant les classes rares
# (usine=4, villa=5 — très peu d'exemples dans le dataset)
RARE_CLASSES = {4, 5}   # usine, villa
RARE_BOOST   = 4.0      # Les images avec ces classes sont tirées 4x plus souvent

labels_dir_path = Path(labels_dir)
sample_weights = []
for img_path in train_imgs:
    fname = os.path.basename(img_path).replace('.jpg', '.txt').replace('.png', '.txt')
    lp = labels_dir_path / fname
    w = 1.0
    if lp.exists() and lp.stat().st_size > 0:
        l = np.loadtxt(lp).reshape(-1, 5)
        if RARE_CLASSES & set(l[:, 0].astype(int)):
            w = RARE_BOOST
    sample_weights.append(w)

sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)
print(f"Images avec classes rares (usine/villa) : {sum(w > 1 for w in sample_weights)} / {len(sample_weights)}")

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler,
                          collate_fn=collate_fn, num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          collate_fn=collate_fn, num_workers=0)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")


Images avec classes rares (usine/villa) : 30 / 115
Train batches: 15 | Val batches: 2


## 3. Configuration du Modèle (Transfer Learning)

Nous utilisons **YOLOv5 Nano**, un modèle très léger et rapide.
1. Charger la configuration du modèle.
2. Initialiser le modèle.
3. Charger les poids pré-entraînés sur COCO (Transfer Learning).
4. Adapter la couche de sortie à notre nombre de classes (6 classes).

In [52]:
# Chemin vers la config du modèle et les données
# On utilise yolov5n.yaml inclus dans le package yolov5
yolov5_path = Path(yolov5.__file__).parent
model_cfg_path = yolov5_path / 'models' / 'yolov5n.yaml'
data_yaml_path = DATA_DIR / 'data.yaml'

# Charger infos dataset
with open(data_yaml_path) as f:
    data_dict = yaml.safe_load(f)
nc = data_dict['nc']  # Nombre de classes (6)
names = data_dict['names']
print(f"Classes ({nc}): {names}")

# Créer le modèle
model = Model(str(model_cfg_path), ch=3, nc=nc).to(DEVICE)

# Charger les poids pré-entraînés
weights_url = "https://github.com/ultralytics/yolov5/releases/download/v7.0/yolov5n.pt"
print(f"Chargement des poids depuis {weights_url}...")
ckpt = torch.hub.load_state_dict_from_url(weights_url, map_location='cpu')

# Exclure les clés incompatibles (ex: anchors, et surtout les têtes de prédiction qui dépendent de nc)
csd = ckpt['model'].float().state_dict()
csd = intersect_dicts(csd, model.state_dict(), exclude=['anchors'])
model.load_state_dict(csd, strict=False)
print("Poids tranférés avec succès.")

Overriding model.yaml nc=80 with nc=6

                 from  n    params  module                                  arguments                     
  0                -1  1      1760  yolov5.models.common.Conv               [3, 16, 6, 2, 2]              
  1                -1  1      4672  yolov5.models.common.Conv               [16, 32, 3, 2]                
  2                -1  1      4800  yolov5.models.common.C3                 [32, 32, 1]                   
  3                -1  1     18560  yolov5.models.common.Conv               [32, 64, 3, 2]                
  4                -1  2     29184  yolov5.models.common.C3                 [64, 64, 2]                   
  5                -1  1     73984  yolov5.models.common.Conv               [64, 128, 3, 2]               


  6                -1  3    156928  yolov5.models.common.C3                 [128, 128, 3]                 
  7                -1  1    295424  yolov5.models.common.Conv               [128, 256, 3, 2]              
  8                -1  1    296448  yolov5.models.common.C3                 [256, 256, 1]                 
  9                -1  1    164608  yolov5.models.common.SPPF               [256, 256, 5]                 
 10                -1  1     33024  yolov5.models.common.Conv               [256, 128, 1, 1]              
 11                -1  1         0  torch.nn.modules.upsampling.Upsample    [None, 2, 'nearest']          
 12           [-1, 6]  1         0  yolov5.models.common.Concat             [1]                           
 13                -1  1     90880  yolov5.models.common.C3                 [256, 128, 1, False]          
 14                -1  1      8320  yolov5.models.common.Conv               [128, 64, 1, 1]               
 15                -1  1         0  t

Classes (6): ['ferme', 'immeuble', 'maison', 'piscine', 'usine', 'villa']


 20                -1  1     74496  yolov5.models.common.C3                 [128, 128, 1, False]          
 21                -1  1    147712  yolov5.models.common.Conv               [128, 128, 3, 2]              
 22          [-1, 10]  1         0  yolov5.models.common.Concat             [1]                           
 23                -1  1    296448  yolov5.models.common.C3                 [256, 256, 1, False]          
 24      [17, 20, 23]  1     14883  yolov5.models.yolo.Detect               [6, [[10, 13, 16, 30, 33, 23], [30, 61, 62, 45, 59, 119], [116, 90, 156, 198, 373, 326]], [64, 128, 256]]
YOLOv5n summary: 214 layers, 1772035 parameters, 1772035 gradients, 4.2 GFLOPs



Chargement des poids depuis https://github.com/ultralytics/yolov5/releases/download/v7.0/yolov5n.pt...
Poids tranférés avec succès.


## 4. Hyperparamètres et Optimiseur

In [53]:
# Hyperparamètres
hyp = {
    'lr0': 0.01,
    'lrf': 0.01,
    'momentum': 0.937,
    'weight_decay': 0.0005,
    'box': 0.05,
    'cls': 0.5,
    'cls_pw': 1.0,
    'obj': 1.0,
    'obj_pw': 1.0,
    'iou_t': 0.20,
    'anchor_t': 4.0,
    'fl_gamma': 1.5,   # Focal Loss : pénalise moins les exemples faciles → aide les classes rares [8]
}

model.hyp = hyp
model.nc = nc
model.names = names

optimizer = optim.SGD(model.parameters(), lr=hyp['lr0'], momentum=hyp['momentum'], weight_decay=hyp['weight_decay'])

lf = one_cycle(1, hyp['lrf'], EPOCHS)
scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lf)

compute_loss = ComputeLoss(model)
print(f"Focal Loss activée avec gamma = {hyp['fl_gamma']}")


Focal Loss activée avec gamma = 1.5


## 5. Boucle d'Entraînement

In [54]:
def train_one_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0
    pbar = tqdm(loader, desc="Training")
    for imgs, targets in pbar:
        imgs = imgs.to(device)
        targets = targets.to(device)
        optimizer.zero_grad()
        pred = model(imgs)
        loss, loss_items = compute_loss(pred, targets)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        pbar.set_postfix({'loss': f"{loss.item():.4f}"})
    return total_loss / len(loader)


def validate(model, loader, device):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for imgs, targets in loader:
            imgs = imgs.to(device)
            targets = targets.to(device)
            out = model(imgs)
            train_out = out[1]
            loss, loss_items = compute_loss(train_out, targets)
            total_loss += loss.item()
    return total_loss / len(loader)


def compute_val_map(model, loader, device, conf_thres=0.25, nms_iou=0.45):
    """Calcule le mAP@0.5 sur le jeu de validation à chaque époque."""
    model.eval()
    stats = []
    iouv = torch.tensor([0.5], device=device)
    with torch.no_grad():
        for imgs, targets in loader:
            imgs = imgs.to(device)
            targets = targets.to(device)
            inference_out = model(imgs)[0]
            preds = non_max_suppression(inference_out, conf_thres, nms_iou)
            for si, det in enumerate(preds):
                gt_mask = targets[:, 0] == si
                gt = targets[gt_mask]
                nl = len(gt)
                tcls = gt[:, 1].tolist() if nl else []
                if len(det) == 0:
                    if nl:
                        stats.append((torch.zeros(0, 1, dtype=torch.bool),
                                      torch.Tensor(), torch.Tensor(), tcls))
                    continue
                correct = torch.zeros(len(det), 1, dtype=torch.bool)
                if nl:
                    x, y, w, h = (gt[:, 2]*IMG_SIZE, gt[:, 3]*IMG_SIZE,
                                  gt[:, 4]*IMG_SIZE, gt[:, 5]*IMG_SIZE)
                    lp = torch.stack([gt[:, 1], x-w/2, y-h/2, x+w/2, y+h/2], 1)
                    iou = box_iou(lp[:, 1:], det[:, :4])
                    xm = torch.where((iou >= 0.5) & (lp[:, 0:1] == det[:, 5]))
                    if xm[0].shape[0]:
                        m = torch.cat((torch.stack(xm, 1),
                                       iou[xm[0], xm[1]][:, None]), 1).cpu().numpy()
                        if xm[0].shape[0] > 1:
                            m = m[m[:, 2].argsort()[::-1]]
                            m = m[np.unique(m[:, 1], return_index=True)[1]]
                        correct[m[:, 1].astype(int)] = True
                stats.append((correct, det[:, 4].cpu(), det[:, 5].cpu(), tcls))
    if not stats:
        return 0.0
    stats_arr = [np.concatenate(x, 0) for x in zip(*stats)]
    if len(stats_arr) and stats_arr[0].any():
        _, _, _, _, _, ap, _ = ap_per_class(
            *stats_arr, plot=False, names=dict(enumerate(names)))
        return float(ap[:, 0].mean())
    return 0.0


# ── Boucle principale ──────────────────────────────────────────────────────────
history = {'train_loss': [], 'val_loss': [], 'val_map': []}
best_val_loss = float('inf')
best_map = 0.0
save_path = 'yolov5n_custom.pt'

# Stratégie de checkpoint :
# - Époques 1-20 (warmup) : sauvegarder sur val_loss (mAP trop bruité sur ~14 images)
# - Époques 21+ : sauvegarder sur mAP@0.5 (plus représentatif de la qualité)
MAP_WARMUP = 20

print(f"Début de l'entraînement : {EPOCHS} époques | fl_gamma={hyp['fl_gamma']} | RARE_BOOST={RARE_BOOST}")
print(f"Checkpoint : val_loss (ép. 1-{MAP_WARMUP}), puis best val_mAP@0.5")

for epoch in range(EPOCHS):
    train_loss = train_one_epoch(model, train_loader, optimizer, DEVICE)
    val_loss   = validate(model, val_loader, DEVICE)
    val_map    = compute_val_map(model, val_loader, DEVICE)
    scheduler.step()

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_map'].append(val_map)

    print(f"Epoch {epoch+1:>3}/{EPOCHS} | "
          f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
          f"Val mAP@0.5: {val_map:.4f}")

    should_save = False
    if epoch < MAP_WARMUP:
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            should_save = True
            reason = f"val_loss={val_loss:.4f}"
    else:
        if val_map > best_map:
            best_map = val_map
            should_save = True
            reason = f"mAP@0.5={val_map:.4f}"

    if should_save:
        ckpt = {
            'epoch': epoch,
            'model': deepcopy(de_parallel(model)).half(),
            'optimizer': optimizer.state_dict(),
            'val_loss': val_loss,
            'val_map': val_map,
            'names': names
        }
        torch.save(ckpt, save_path)
        print(f"  ✓ Modèle sauvegardé ({reason}) → {os.path.abspath(save_path)}")

print(f"\nEntraînement terminé. Meilleur mAP@0.5 val : {best_map:.4f}")
with open("training_history.json", "w") as f:
    json.dump(history, f)
print("Historique sauvegardé dans training_history.json")


Début de l'entraînement : 100 époques | fl_gamma=1.5 | RARE_BOOST=4.0
Checkpoint : val_loss (ép. 1-20), puis best val_mAP@0.5


Training: 100%|██████████| 15/15 [00:22<00:00,  1.52s/it, loss=0.5378]


Epoch   1/100 | Train Loss: 1.7600 | Val Loss: 1.4150 | Val mAP@0.5: 0.0000
  ✓ Modèle sauvegardé (val_loss=1.4150) → c:\Users\olivi\Documents\GitHub\SISE_satelitar_identifier\notebooks\yolov5n_custom.pt


Training: 100%|██████████| 15/15 [00:22<00:00,  1.47s/it, loss=0.5527]


Epoch   2/100 | Train Loss: 1.3919 | Val Loss: 1.2929 | Val mAP@0.5: 0.0000
  ✓ Modèle sauvegardé (val_loss=1.2929) → c:\Users\olivi\Documents\GitHub\SISE_satelitar_identifier\notebooks\yolov5n_custom.pt


...

Epoch  98/100 | Train Loss: 0.3963 | Val Loss: 1.0181 | Val mAP@0.5: 0.2328


Training: 100%|██████████| 15/15 [00:22<00:00,  1.48s/it, loss=0.0924]


Epoch  99/100 | Train Loss: 0.3911 | Val Loss: 1.0171 | Val mAP@0.5: 0.2324


Training: 100%|██████████| 15/15 [00:22<00:00,  1.53s/it, loss=0.1701]


Epoch 100/100 | Train Loss: 0.3625 | Val Loss: 1.0209 | Val mAP@0.5: 0.2321

Entraînement terminé. Meilleur mAP@0.5 val : 0.3123
Historique sauvegardé dans training_history.json


## 6. Visualisation de l'Histoire d'Entraînement

Affichons l'évolution de la fonction de perte (Loss) pour les jeux d'entraînement et de validation.

In [55]:
plt.figure(figsize=(10, 6))
plt.plot(history['train_loss'], label='Train Loss')
plt.plot(history['val_loss'], label='Validation Loss')
plt.xlabel('Époques')
plt.ylabel('Loss')
plt.title('Courbes d\'apprentissage')
plt.legend()
plt.grid(True)
plt.savefig('training_history.png')
plt.show()